In [ ]:
#This file will be used to write the inference metrics obtained from footnet_inference.py
import sys, os
sys.path.insert(0, "/Users/abhinavarora/Desktop/CadenceCV/ml/src")
from inference.footnet_inference import run_inference
import pickle
import numpy as np
import scipy.signal

# Load the cached YOLO output — run main.py or footnet_inference_test.ipynb first to create this.
# frame_count, duration, hip/ankle/knee arrays come from compute_all_metrics (YOLO),
# not from run_inference, so they live in the cache alongside frame_by_frame_data.
cache_path = "/Users/abhinavarora/Desktop/CadenceCV/ml/data/keypoints_cache.pkl"
with open(cache_path, "rb") as f:
    cache = pickle.load(f)

frame_by_frame_data = cache["frame_by_frame_data"]
frame_count = cache["frame_count"]
duration = cache["duration"]
right_hip_arr = cache["right_hip_arr"]
left_hip_arr = cache["left_hip_arr"]
right_ankle_arr = cache["right_ankle_arr"]
left_ankle_arr = cache["left_ankle_arr"]
right_knee_arr = cache["right_knee_arr"]
left_knee_arr = cache["left_knee_arr"]

# Run FootNet LSTM inference on the cached per-frame metrics (YOLO does not re-run here)
strikefoot_frames, frame_to_pred_dict = run_inference(frame_by_frame_data)

In [ ]:
#This conversion is required for accruate metric reporting
#The most accurate converter will take a 
def pixel_to_actual_units(height, img_dir):
    pass

In [ ]:
#Clauclating the average time in air per gait cycle and average time in ground contact
def calculate_metrics():
    #Grouping pairs of 1s and 0s from the frame_to_pred_dict to obtain a gait cycle
    #Must iterate in sorted frame order 
    frame_preds = [frame_to_pred_dict[f] for f in sorted(frame_to_pred_dict.keys())]

    #(i+1)th - ith entry
    diff = np.diff(frame_preds)
    #contact:1, non-contact: 0. 
    #So, non_contact AFTER contact = 0 - 1 = -1 (takeoff frame)
    takeoff_frames = np.where(diff == -1)[0]
    #So, contact AFTER non-contact = 1 - 0 = 1 (Strikefoot frame)
    landing_frames = np.where(diff == 1)[0]

    flight_times = []
    gct_times = []
    for takeoff in takeoff_frames:
        #Find the next landing after this takeoff. Finding all landings initially to check if there even is a landing 
        #after the take off frames.
        next_landings = landing_frames[landing_frames > takeoff]
        if len(next_landings) > 0:
            next_landing = next_landings[0]
            flight_times.append(next_landing - takeoff)
    
    for landing in landing_frames:
        next_takeoffs = takeoff_frames[takeoff_frames > landing]
        if len(next_takeoffs) > 0:
            next_takeoff = next_takeoffs[0]  # was next_takeoff[0] — typo, wrong variable
            gct_times.append(next_takeoff - landing)

    #Divide by frame count to obtain metrics in seconds
    average_flight_time = (np.mean(flight_times) if flight_times else 0) / frame_count
    average_gct = (np.mean(gct_times) if gct_times else 0) / frame_count

    #Cadence is the steps per second
    cadence = len(strikefoot_frames) / duration

    #Vertical oscillation: 
    # 1) Calculate the hip's mean Y coordinate (mean of right and left hip Y) over all frames
    # 2) Apply the Savitz-Golay smoothing filter (finetune the window size)
    # 3) Separate out the smoothed array based on each gait cycle: (continuous pairs of 0s and 1s)
    # 4) Obtain the vertical difference between each peak and trough (since pixel coordinates go downward)

    # Index 1 is the Y coordinate (vertical axis in pixel space)
    n = min(len(right_hip_arr), len(left_hip_arr))
    mean_hip_y_coords = []
    for i in range(n):
        mean_hip_y_coords.append(np.mean([right_hip_arr[i][1], left_hip_arr[i][1]]))
    
    #The window length is the number of coefficients for smoothing
    #The poly order is the degree of the filtering polynomial
    #Finetune these accordingly
    mean_hip_y_coords = scipy.signal.savgol_filter(mean_hip_y_coords, window_length=4, polyorder=3)

    #Extracting gait cycles: Pictorial explanation in the next cell:
    changes = np.where(np.diff(frame_preds) == -1)[0] + 1
    #Pairs of -1 is one gait cycle
    cycles = []
    cycle = []
    for i in changes:
        cycle.append(i)
        if len(cycle) == 2:
            cycles.append(cycle.copy())
            cycle = []
    
    oscillations = []
    for c in cycles:
        start = c[0]
        end = c[1]
        peak = max(mean_hip_y_coords[start:end])
        trough = min(mean_hip_y_coords[start:end])
        oscillations.append(peak - trough)
    
    avg_oscillation = np.mean(oscillations) if oscillations else 0
    
    #Calculating stride length:

In [ ]:
#Classifying overstrides: 
# a) Shin Test: Checking the angle between the vertical and the shin bone. (ankle to knee). If angle > 5 degrees, then the person is overstriding
# b) Center of Mass test: Seeing if the ankle goes beyond the Center of Mass (mean of the hip's coordinates):
# 1) 0 - 5 cm beyond the COM: Optimal
# 2) 5 - 10 cm beyond the COM: Mild Overstriding
# 3) 10 - 15 cm beyond the COM: Severe Overstriding


#Placeholders until there is a conversion system from pixel to actual height
OFFSET_NEUTRAL_MAX_PX  = 30   # placeholder for < 5cm
OFFSET_MILD_MAX_PX     = 60   # placeholder for 5-10cm, > 60px. Severe overstride

TIBIAL_NEUTRAL_MAX_RAD = 0.25  # ~14° forward tilt
TIBIAL_MILD_MAX_RAD    = 0.50  # ~29° forward tilt, > 0.50 rad. Severe overstride


def _classify_offset(offset_px):
    if offset_px < OFFSET_NEUTRAL_MAX_PX:
        return "neutral"
    elif offset_px < OFFSET_MILD_MAX_PX:
        return "mild_overstride"
    else:
        return "severe_overstride"


def _classify_tibial(tibial_rad):
    # abs() because the angle can be negative depending on running direction
    angle = abs(tibial_rad)
    if angle < TIBIAL_NEUTRAL_MAX_RAD:
        return "neutral"
    elif angle < TIBIAL_MILD_MAX_RAD:
        return "mild_overstride"
    else:
        return "severe_overstride"


def classify_overstriding(strikefoot_frames, frame_by_frame_data):
    #Neutral and overstriding are classified if both tests pass or fail respectively
    neutral = []
    overstriding = []
    #Frames appended here when both tests produce different results. However, it is known that the COM test is more accurate
    conflict = []

    for frame in strikefoot_frames:
        entry = frame_by_frame_data[frame]

        test1 = _classify_offset(entry["ankle_hip_horizontal_offset"])
        test2 = _classify_tibial(entry["tibial_angle"])

        both_neutral = (test1 == "neutral") and (test2 == "neutral")
        both_overstride = (test1 != "neutral") and (test2 != "neutral")

        if both_neutral:
            neutral.append(frame)
        elif both_overstride:
            overstriding.append(frame)
        else:
            # The two tests disagree
            conflict.append({
                "frame": frame,
                "Ankle_Hip_Offset_Test": test1,
                "Tibial_Angle_Test": test2,
            })

    return neutral, overstriding, conflict

#Will be used for analysis later on
neutral_frames, overstriding_frames, conflict_frames = classify_overstriding(
    strikefoot_frames, frame_by_frame_data
)

print(f"Neutral ({len(neutral_frames)} frames): {neutral_frames}")
print(f"Overstride ({len(overstriding_frames)} frames): {overstriding_frames}")
print(f"Conflict ({len(conflict_frames)} frames):")
for c in conflict_frames:
    print(f"  frame {c['frame']}: {c['Ankle_Hip_Offset_Test']} | tibial → {c['Tibial_Angle_Test']}")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage
imgplot = plt.imread("/Users/abhinavarora/Desktop/CadenceCV/docs/gait_cycle_extraction.jpg")
#Rotating the initial image to be upright
rotated_image = ndimage.rotate(imgplot, -90, reshape=True)
plt.axis('off')
imgplot = plt.imshow(rotated_image)
